# 1 · Reading your dataset

This guide takes a public dataset from its download to a WarpRec `Dataset`, the
object every model trains on and every evaluation reads. It covers:

- preparing a file WarpRec can read, and the shapes of file it accepts;
- the `LocalReader`: separators, headers, column names and types, parquet,
  and the two dataframe backends;
- building a `Dataset` and reading what it holds: id mappings, the interaction
  matrix and the `info()` every model is built from;
- loading data that is already split, including validation folds, and writing
  a split out so it can be loaded that way later.

Every step is done twice: through the Python API, then through the `reader`
section of a configuration file, which is what the pipelines run.

**Requirements:** `pip install warprec jupyter`. Runs in under a minute; it
downloads MovieLens-100K (5 MB) into `guides/data/`.

In [1]:
import io
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

DATA = Path("../data")
RUNS = Path("../runs/01-reading-data")
RUNS.mkdir(parents=True, exist_ok=True)

## Preparing the data

WarpRec reads one row per interaction. The only required columns are the user
and the item; a `rating` is used as the interaction's strength when the dataset
is read as explicit, and a `timestamp` is what temporal splits and sequential
models order by. MovieLens-100K ships its ratings as `u.data`: tab separated,
no header, in exactly that order.

In [2]:
url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
folder = DATA / "ml-100k"
if not folder.exists():
    with urllib.request.urlopen(url) as response:
        zipfile.ZipFile(io.BytesIO(response.read())).extractall(DATA)

print((folder / "u.data").read_text().splitlines()[:3])

['196\t242\t3\t881250949', '186\t302\t3\t891717742', '22\t377\t1\t878887116']


Giving the file a header makes it self-describing, so the configuration does
not have to list the columns. The other guides read this `ratings.tsv`; the
helper `guides/guide_data.py` repeats exactly these steps so that every guide
can be run on its own.

In [3]:
ratings = pd.read_csv(
    folder / "u.data", sep="\t", names=["user_id", "item_id", "rating", "timestamp"]
)
ratings.to_csv(folder / "ratings.tsv", sep="\t", index=False)
print(
    f"{len(ratings):,} interactions, {ratings.user_id.nunique()} users, "
    f"{ratings.item_id.nunique()} items"
)
ratings.head(3)

100,000 interactions, 943 users, 1682 items


,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116


## The reader

A reader turns a file into a dataframe with WarpRec's column names. It returns
a [Narwhals](https://narwhals-dev.github.io/narwhals/) frame, a thin layer over
Polars (the default backend) or pandas, so the rest of WarpRec does not care
which of the two produced it.

In [4]:
from warprec.data.reader import LocalReader

reader = LocalReader()
data = reader.read_tabular(local_path=str(folder / "ratings.tsv"), sep="\t")
print(type(data).__name__, "over", type(data.to_native()).__module__.split(".")[0])
data.to_native().head(3)

DataFrame over polars


user_id,item_id,rating,timestamp
i64,i64,i64,i64
196,242,3,881250949
186,302,3,891717742
22,377,1,878887116


Called on its own, the reader keeps the types it infers:

In [5]:
data.schema

Schema([('user_id', Int64),
        ('item_id', Int64),
        ('rating', Int64),
        ('timestamp', Int64)])

A file without a header needs its column names, in the order the columns
appear, and `dtypes` casts any column as it is read. Here the original `u.data`
is read directly, with the types a configuration uses unless told otherwise
(`int32` ids and timestamps, `float32` ratings), which halves the memory a large
file takes:

In [6]:
headerless = reader.read_tabular(
    local_path=str(folder / "u.data"),
    sep="\t",
    header=False,
    column_names=["user_id", "item_id", "rating", "timestamp"],
    dtypes={
        "user_id": "int32",
        "item_id": "int32",
        "rating": "float32",
        "timestamp": "int32",
    },
)
headerless.schema

Schema([('user_id', Int32),
        ('item_id', Int32),
        ('rating', Float32),
        ('timestamp', Int32)])

Parquet keeps its own types, so it needs neither a separator nor names. The
pandas backend is selected on the reader; everything downstream is unchanged.

In [7]:
ratings.to_parquet(RUNS / "ratings.parquet", index=False)

pandas_reader = LocalReader(backend="pandas")
from_parquet = pandas_reader.read_parquet(local_path=str(RUNS / "ratings.parquet"))
print(type(from_parquet.to_native()).__module__.split(".")[0], from_parquet.shape)

pandas (100000, 4)


A file the reader cannot find comes back **empty** rather than raising, so a
typo in a path shows up as an empty dataset further on. Check the size of what
you read.

In [8]:
reader.read_tabular(local_path=str(folder / "no_such_file.tsv")).shape

(0, 0)

### Columns with other names

A file whose columns are called something else is mapped onto WarpRec's names
with `labels`. Read through a configuration, the columns below behave exactly
like `user_id`, `item_id`, `rating` and `timestamp`.

In [9]:
renamed = ratings.rename(
    columns={
        "user_id": "userId",
        "item_id": "movieId",
        "rating": "stars",
        "timestamp": "when",
    }
)
renamed.to_csv(RUNS / "renamed.tsv", sep="\t", index=False)
print(Path("configs/labels.yml").read_text())

# The same file with other column names, mapped onto WarpRec's.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../runs/01-reading-data/renamed.tsv
  rating_type: explicit
  labels:
    user_id_label: userId
    item_id_label: movieId
    rating_label: stars
    timestamp_label: when
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
models:
  Pop: {}
evaluation:
  top_k: [10]
  metrics: [nDCG]



## The same thing from a configuration

The pipelines never call the reader themselves: they read the `reader` section
of the configuration. This is the smallest configuration that reads
`ratings.tsv`; a `splitter` is required because a single file has to be split
before a model can be evaluated on it (guide 3 covers splitting).

In [10]:
print(Path("configs/reader-dataset.yml").read_text())

# Read one file and split it. Paths are relative to the guide's folder.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  sep: "\t"
  header: true
  rating_type: explicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
# Every configuration names at least one model and how to evaluate it.
# Guides 9 and 12 cover these two sections.
models:
  Pop: {}
evaluation:
  top_k: [10]
  metrics: [nDCG]



`initialize_datasets` is the call every pipeline makes with that section: it
reads, filters, splits and builds the datasets. It returns the main dataset
(train and test), a validation dataset when one is configured, and a list of
fold datasets under cross-validation.

In [11]:
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory
from warprec.utils.callback import WarpRecCallback
from warprec.utils.config.config import load_design_configuration

config = load_design_configuration("configs/reader-dataset.yml")
main, validation, folds = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
print(
    main.info()["n_users"],
    "users,",
    main.info()["n_items"],
    "items;",
    "validation:",
    validation,
    "folds:",
    len(folds),
)

08-10-2026 12:22:00 - WarpRec - 📝 Reading design configuration file in: configs/reader-dataset.yml


08-10-2026 12:22:00 - WarpRec - 📝 Reading process completed correctly.


08-10-2026 12:22:00 - WarpRec - 📝 Starting test splitting process with temporal_holdout splitting strategy.


08-10-2026 12:22:00 - WarpRec - 📝 Test splitting completed in : 0.03s


08-10-2026 12:22:00 - WarpRec - ✅ Splitting process over in 0.03s.


...

08-10-2026 12:22:00 - WarpRec - 📝 ------------------------------------------------------------------------


08-10-2026 12:22:00 - WarpRec - 📝 --------------------------------Test set--------------------------------


08-10-2026 12:22:00 - WarpRec - 📊 Number of users: 943      Number of items: 1335      Transactions: 10401


08-10-2026 12:22:00 - WarpRec - 📝 ------------------------------------------------------------------------


943 users, 1648 items; validation: None folds: 0


The log reports every step: what was read, how the split went, how many test
rows were dropped because their user or item never appears in training, and
the size of each set.

WarpRec logs through [loguru](https://loguru.readthedocs.io/), so a notebook can
silence it with `logger.disable("warprec")` and bring it back with
`logger.enable("warprec")`. The rest of this guide keeps it off except where the
log is the point.

In [12]:
from loguru import logger

logger.disable("warprec")

The renamed file reads to the same dataset through `configs/labels.yml`:

In [13]:
config = load_design_configuration("configs/labels.yml")
relabelled, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
relabelled.train_set.get_sparse().nnz == main.train_set.get_sparse().nnz

True

## The `Dataset`

A `Dataset` is built from a training frame and an evaluation frame. The split
below holds out each user's most recent 10% of interactions; guide 3 covers
every strategy.

In [14]:
from warprec.data import Dataset
from warprec.data.schema import SplitSpec
from warprec.data.splitting import Splitter

train, _, test = Splitter().split_transaction(
    data, test=SplitSpec(strategy="temporal_holdout", ratio=0.1)
)
dataset = Dataset(
    train_data=train,
    eval_data=test,
    rating_type="explicit",
    rating_label="rating",
    timestamp_label="timestamp",
)

Users and items are given contiguous indices, from 0, in the order of their
original ids. **Only the training split defines them**: an item that appears
only in the test split cannot be scored, so its test rows are dropped (the
cold-start protocols in guide 13 are the exception). Index `n_items` is never
an item: models use it as padding.

In [15]:
user_map, item_map = dataset.get_mappings()
print(dataset.get_dims(), "users x items")
print("raw user 196 -> index", user_map[196], "| raw item 242 -> index", item_map[242])

(943, 1648) users x items
raw user 196 -> index 195 | raw item 242 -> index 241


`info()` is what every model is constructed from. Its entries grow with what
the dataset carries: side information, contexts, a knowledge graph or
multimodal features each add their own (guides 4–7).

In [16]:
{
    k: (v if not isinstance(v, dict) else f"<{len(v)} entries>")
    for k, v in dataset.info().items()
}

{'n_items': 1648,
 'n_users': 943,
 'n_features': 0,
 'item_mapping': '<1648 entries>',
 'user_mapping': '<943 entries>'}

The splits are held as `Interactions`: a sparse user × item matrix over the
whole catalogue. `rating_type` decides what the matrix holds. `explicit` keeps
the ratings, `implicit` stores a 1 for every interaction.

In [17]:
explicit = dataset.train_set.get_sparse()
implicit = Dataset(
    train_data=train, eval_data=test, rating_type="implicit"
).train_set.get_sparse()
pd.DataFrame(
    {
        "shape": [explicit.shape, implicit.shape],
        "stored": [explicit.nnz, implicit.nnz],
        "values": [sorted(set(explicit.data)), sorted(set(implicit.data))],
    },
    index=["explicit", "implicit"],
)

,shape,stored,values
explicit,"(943, 1648)",89561,"[1, 2, 3, 4, 5]"
implicit,"(943, 1648)",89561,[1.0]


## Data that is already split

With `loading_strategy: split`, WarpRec reads splits you made yourself instead
of splitting a single file. It expects one directory holding:

- `train.tsv` and `test.tsv`;
- optionally `validation.tsv`, for a single validation set;
- or, for cross-validation, numbered folders `1/`, `2/`, … each holding its own
  `train.tsv` and `validation.tsv`.

MovieLens-100K ships ready-made splits; `u1.base` / `u1.test` is one of five
80/20 partitions. Written in that layout:

In [18]:
def write_split(directory: Path, **frames: pd.DataFrame) -> None:
    """Write named frames as <name>.tsv files under directory."""
    for name, frame in frames.items():
        path = directory / f"{name}.tsv"
        path.parent.mkdir(parents=True, exist_ok=True)
        frame.to_csv(path, sep="\t", index=False)


def movielens_file(name: str) -> pd.DataFrame:
    return pd.read_csv(folder / name, sep="\t", names=list(ratings.columns))


u1_train, u1_test = movielens_file("u1.base"), movielens_file("u1.test")
write_split(RUNS / "split-u1", train=u1_train, test=u1_test)
sorted(str(p.relative_to(RUNS)) for p in (RUNS / "split-u1").rglob("*.tsv"))

['split-u1/test.tsv', 'split-u1/train.tsv']

In [19]:
print(Path("configs/reader-split.yml").read_text())

# Read a ready-made train/test split: split/train.tsv and split/test.tsv.
reader:
  loading_strategy: split
  data_type: transaction
  reading_method: local
  rating_type: explicit
  split:
    local_path: ../runs/01-reading-data/split-u1
    ext: .tsv
    sep: "\t"
    header: true
models:
  Pop: {}
evaluation:
  top_k: [10]
  metrics: [nDCG]



In [20]:
logger.enable("warprec")
config = load_design_configuration("configs/reader-split.yml")
presplit, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
print(
    "train",
    presplit.train_set.get_sparse().nnz,
    "| test",
    presplit.eval_set.get_sparse().nnz,
    "of",
    len(u1_test),
    "rows",
)

08-10-2026 12:22:00 - WarpRec - 📝 Reading design configuration file in: configs/reader-split.yml


08-10-2026 12:22:00 - WarpRec - 📝 Reading process completed correctly.


08-10-2026 12:22:00 - WarpRec - 📝 Starting reading split process from: ../runs/01-reading-data/split-u1 using polars


08-10-2026 12:22:00 - WarpRec - ✅ Reading process completed successfully.


08-10-2026 12:22:00 - WarpRec - 📢 Test set was not aligned with the training set. Filtered out 32 transactions.


...

08-10-2026 12:22:00 - WarpRec - 📝 ------------------------------------------------------------------------


08-10-2026 12:22:00 - WarpRec - 📝 --------------------------------Test set--------------------------------


08-10-2026 12:22:00 - WarpRec - 📊 Number of users: 459      Number of items: 1378      Transactions: 19968


08-10-2026 12:22:00 - WarpRec - 📝 ------------------------------------------------------------------------


train 80000 | test 19968 of 20000 rows


The test set holds 32 rows fewer than the file: their item never appears in
`u1.base`, so they were aligned away, as the log says.

Validation folds follow the same layout one level down. Here the training part
of `ua.base` / `ua.test` (MovieLens' other ready-made split, ten test ratings per
user) is cut into three folds of its own, each fold validating on a different
third of every user's history:

In [21]:
ua_train, ua_test = movielens_file("ua.base"), movielens_file("ua.test")
fold_of = ua_train.groupby("user_id").cumcount() % 3
write_split(RUNS / "split-ua-folds", train=ua_train, test=ua_test)
for fold in range(3):
    write_split(
        RUNS / "split-ua-folds" / str(fold + 1),
        train=ua_train[fold_of != fold],
        validation=ua_train[fold_of == fold],
    )
sorted(str(p.relative_to(RUNS)) for p in (RUNS / "split-ua-folds").rglob("*.tsv"))

['split-ua-folds/1/train.tsv',
 'split-ua-folds/1/validation.tsv',
 'split-ua-folds/2/train.tsv',
 'split-ua-folds/2/validation.tsv',
 'split-ua-folds/3/train.tsv',
 'split-ua-folds/3/validation.tsv',
 'split-ua-folds/test.tsv',
 'split-ua-folds/train.tsv']

In [22]:
logger.disable("warprec")
config = load_design_configuration("configs/reader-folds.yml")
main_ua, validation_ua, fold_datasets = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
pd.DataFrame(
    [
        {
            "dataset": name,
            "train": d.train_set.get_sparse().nnz,
            "evaluation": d.eval_set.get_sparse().nnz,
        }
        for name, d in [("main", main_ua)]
        + [(f"fold {i + 1}", d) for i, d in enumerate(fold_datasets)]
    ]
)

,dataset,train,evaluation
0,main,90570,9428
1,fold 1,60052,30043
2,fold 2,60386,30121
3,fold 3,60702,29789


Each fold trains on its own `train.tsv` and is validated on its own
`validation.tsv`; the main dataset trains on the full `train.tsv` and is tested
on `test.tsv`. That is what the training pipeline tunes on and then reports.

## Writing a split

A split WarpRec computed can be written in that same layout, so later runs (or
other frameworks) use exactly the same partition. In a configuration this is
`writer.save_split: true`; through the API it is the writer's
`write_tabular_split`.

In [23]:
from warprec.data.writer import LocalWriter

train, validation_folds, test = Splitter().split_transaction(
    data,
    test=SplitSpec(strategy="temporal_holdout", ratio=0.1),
    validation=SplitSpec(strategy="k_fold_cross_validation", folds=3),
)
main_written = Dataset(
    train_data=train, eval_data=test, rating_type="explicit", rating_label="rating"
)
folds_written = [
    Dataset(
        train_data=t,
        eval_data=v,
        rating_type="explicit",
        rating_label="rating",
        evaluation_set="Validation",
    )
    for t, v in validation_folds
]
writer = LocalWriter(dataset_name="ml-100k", local_path=str(RUNS / "written"))
writer.write_tabular_split(main_written, None, folds_written)
sorted(
    str(p.relative_to(writer.experiment_split_path))
    for p in Path(writer.experiment_split_path).rglob("*.tsv")
)

['1/train.tsv',
 '1/validation.tsv',
 '2/train.tsv',
 '2/validation.tsv',
 '3/train.tsv',
 '3/validation.tsv',
 'test.tsv',
 'train.tsv']

Read back with `loading_strategy: split`, it gives the same datasets:

In [24]:
back = LocalReader().read_tabular_split(
    local_path=writer.experiment_split_path, column_names=None, dtypes=None
)
train_back, folds_back, test_back = back
print(
    len(train_back) == len(train),
    len(test_back) == len(test),
    [len(v) for _, v in folds_back] == [len(v) for _, v in validation_folds],
)

True True True


## Summary of the `reader` section

| key | what it does |
|---|---|
| `loading_strategy` | `dataset` reads one file and splits it; `split` reads ready-made splits from `split.local_path` |
| `local_path`, `sep`, `header`, `file_format` | where and how to read the single file; `file_format` is `tabular` or `parquet` |
| `rating_type` | `explicit` keeps the rating as the interaction's value, `implicit` stores 1 |
| `labels` | the names your columns have, mapped onto WarpRec's |
| `dtypes` | the type each core column is read as |
| `split` | `local_path`, `ext`, `sep`, `header`, `file_format` of a ready-made split |

The full list, including side information, contexts, knowledge graphs, multimodal
features and clusters, is on the [Reader configuration](https://warprec.readthedocs.io/en/latest/configuration/reader/)
page; guides 4–7 cover those sections.